In [ ]:
# Cell 1 - imports
import json
import os
from pathlib import Path

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from IPython.display import Video, display
from PIL import Image, ImageDraw


In [ ]:
# Display and detection helpers
VIDEO_DIR = Path("generated_videos")
VIDEO_DIR.mkdir(exist_ok=True)

def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(path, frames.astype(np.uint8), fps=fps)
    return path

def show_video(frames, name, fps=20, embed=True):
    try:
        path = save_video(frames, VIDEO_DIR / name, fps=fps)
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
        return path
    except Exception as exc:
        print(f"Video display skipped: {type(exc).__name__}: {exc}")
        return None

def show_video_file(path, embed=True):
    path = Path(path)
    display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path

def bbox_from_mask(mask, min_area=50):
    mask = mask.astype(bool)
    visited = np.zeros(mask.shape, dtype=bool)
    boxes = []
    height, width = mask.shape

    ys, xs = np.nonzero(mask)
    for y0, x0 in zip(ys, xs):
        if visited[y0, x0] or not mask[y0, x0]:
            continue

        stack = [(int(y0), int(x0))]
        visited[y0, x0] = True
        x_min = x_max = int(x0)
        y_min = y_max = int(y0)
        area = 0

        while stack:
            y, x = stack.pop()
            area += 1
            x_min = min(x_min, x)
            x_max = max(x_max, x)
            y_min = min(y_min, y)
            y_max = max(y_max, y)

            for ny in (y - 1, y, y + 1):
                for nx in (x - 1, x, x + 1):
                    if ny == y and nx == x:
                        continue
                    if 0 <= ny < height and 0 <= nx < width and mask[ny, nx] and not visited[ny, nx]:
                        visited[ny, nx] = True
                        stack.append((ny, nx))

        if area >= min_area:
            boxes.append((x_min, y_min, x_max + 1, y_max + 1, float(area)))

    return boxes

def draw_boxes(ax, boxes, color="lime", labels=None):
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2, edgecolor=color)
        ax.add_patch(rect)
        if labels:
            ax.text(x1, max(0, y1 - 4), labels[i], color=color, fontsize=9, weight="bold")

def show_detection_frame(frames, frame_idx, boxes, title, labels=None):
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frames[frame_idx])
    draw_boxes(ax, boxes, labels=labels)
    ax.set_title(title)
    ax.axis("off")
    plt.show()

def plot_detection_metric(values, title, ylabel="objects"):
    plt.figure(figsize=(8, 3))
    plt.plot(values)
    plt.title(title)
    plt.xlabel("frame")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:
# Internet video helpers for independent visual problems 4 and 5
def download_video(url, path):
    import urllib.request
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    return path

def read_video(path, max_frames=None, stride=1):
    import cv2
    cap = cv2.VideoCapture(str(path))
    frames_out = []
    frame_no = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_no % stride == 0:
            frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames_out) >= max_frames:
                break
        frame_no += 1
    cap.release()
    if not frames_out:
        raise RuntimeError(f"No frames read from {path}")
    return np.stack(frames_out)


# Visual Problems


## 1 - Lunar Lander


In [ ]:
# LunarLander setup - use kernel: Python (Action Inference)
import sys
import gymnasium as gym
from gymnasium.envs.box2d.lunar_lander import heuristic

ACTION_NAMES = {
    0: "noop",
    1: "left_engine",
    2: "main_engine",
    3: "right_engine",
}

print(sys.executable)
print(gym.__version__)


In [ ]:
# LunarLander rollout
def rollout_lunar_lander(policy, seed=0, max_steps=300, render_mode="rgb_array"):
    env = gym.make("LunarLander-v3", render_mode=render_mode)
    obs, info = env.reset(seed=seed)
    observations, actions, rewards, frames, infos = [], [], [], [], []

    for t in range(max_steps):
        action = int(policy(env.unwrapped, obs))
        next_obs, reward, terminated, truncated, info = env.step(action)
        observations.append(obs.copy())
        actions.append(action)
        rewards.append(float(reward))
        infos.append(info)
        frames.append(env.render())
        obs = next_obs
        if terminated or truncated:
            break

    env.close()
    return {
        "observations": np.asarray(observations, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.int64),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "frames": np.stack(frames),
        "infos": infos,
    }

rollout = rollout_lunar_lander(heuristic, seed=0)
frames = rollout["frames"]
actions = rollout["actions"]
observations = rollout["observations"]

print("frames:", frames.shape)
print("observations:", observations.shape)
print("actions:", actions.shape)
print("action counts:", {ACTION_NAMES[i]: int((actions == i).sum()) for i in ACTION_NAMES})


In [ ]:
frame_idx = min(10, len(frames) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames[frame_idx])
plt.title(f"t={frame_idx}, action={actions[frame_idx]} ({ACTION_NAMES[int(actions[frame_idx])]})")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames, "1_lunar_lander.mp4", fps=30)


## 2 - Car Racing


In [ ]:
# Install once if Box2D is missing
!pip install gymnasium[box2d] -q


In [ ]:
# CarRacing with random driving
import gymnasium as gym

env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
obs, _ = env.reset(seed=0)

frames_car = []
for t in range(300):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    frames_car.append(env.render())
    if terminated or truncated:
        break

env.close()
frames_car = np.stack(frames_car)
frames_car.shape


In [ ]:
car_frame_idx = min(40, len(frames_car) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames_car[car_frame_idx])
plt.title(f"CarRacing frame {car_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_car, "2_car_racing.mp4", fps=30)


## 3 - Recorded Traffic With People


In [ ]:
# Load recorded traffic video
import cv2
import urllib.request

if not Path("traffic.avi").exists():
    url = "https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/cv/video/768x576.avi"
    urllib.request.urlretrieve(url, "traffic.avi")

cap = cv2.VideoCapture("traffic.avi")
frames_medium = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_medium.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()
frames_medium = np.stack(frames_medium)
frames_medium.shape


In [ ]:
traffic_frame_idx = min(1, len(frames_medium) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_medium[traffic_frame_idx])
plt.title(f"Recorded traffic frame {traffic_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_medium, "3_recorded_traffic_people.mp4", fps=20)


## 4 - Random YouTube Driving Scene


In [ ]:
# Problem 4: random 3-minute YouTube driving clip
PROBLEM_4_URL = "https://www.youtube.com/watch?v=7EovwWQIvBo"
problem_4_path = Path("external_videos/problem_4_youtube_random.mp4")
if not problem_4_path.exists():
    raise FileNotFoundError(f"Missing {problem_4_path}. Download it with yt-dlp first.")

# Display the full downloaded 3-minute clip.
show_video_file(problem_4_path)

# Use a frame sample for plotting and object detection so the notebook stays responsive.
frames_problem4 = read_video(problem_4_path, max_frames=900, stride=6)
problem4_frame_idx = min(30, len(frames_problem4) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem4[problem4_frame_idx])
plt.title(f"Problem 4 sampled frame {problem4_frame_idx}")
plt.axis("off")
plt.show()


## 5 - Hard Vehicle-Crowd Interaction


In [ ]:
# Problem 5: hardest 3-minute vehicle-crowd YouTube clip
PROBLEM_5_URL = "https://www.youtube.com/watch?v=7HaJArMDKgI"
problem_5_path = Path("external_videos/problem_5_youtube_hardest.mp4")
if not problem_5_path.exists():
    raise FileNotFoundError(f"Missing {problem_5_path}. Download it with yt-dlp first.")

# Display the full downloaded 3-minute clip.
show_video_file(problem_5_path)

# Use a frame sample for plotting and object detection so the notebook stays responsive.
frames_problem5 = read_video(problem_5_path, max_frames=900, stride=6)
problem5_frame_idx = min(40, len(frames_problem5) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem5[problem5_frame_idx])
plt.title(f"Problem 5 sampled frame {problem5_frame_idx}")
plt.axis("off")
plt.show()


# Object Detection


In [ ]:
# YOLO setup and display utilities. Run this once before the detection subsections.
!pip install ultralytics -q

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

def run_yolo(frames_batch):
    results = model(list(frames_batch), stream=True, verbose=False)
    all_detections = []
    for r in results:
        boxes = r.boxes
        frame_dets = [
            (int(c), float(conf), *map(float, xyxy))
            for c, conf, xyxy in zip(boxes.cls, boxes.conf, boxes.xyxy)
        ]
        all_detections.append(frame_dets)
    return all_detections

def yolo_dets(frame_dets, conf_min=0.25):
    boxes, labels = [], []
    for class_id, conf, x1, y1, x2, y2 in frame_dets:
        if conf >= conf_min:
            boxes.append((x1, y1, x2, y2, conf))
            labels.append(f"{model.names[int(class_id)]} {conf:.2f}")
    return boxes, labels

def yolo_detection_counts(all_detections, conf_min=0.25):
    return np.asarray([sum(1 for _, conf, *_ in frame_dets if conf >= conf_min) for frame_dets in all_detections], dtype=int)


## 1 - Lunar Lander Detection


In [ ]:
# YOLO decides what, if anything, it recognizes in LunarLander. No manual class labels.
if "detections_lunar" not in globals():
    detections_lunar = run_yolo(frames)
if "lunar_yolo_counts" not in globals():
    lunar_yolo_counts = yolo_detection_counts(detections_lunar)

boxes, labels = yolo_dets(detections_lunar[frame_idx])
print(labels if labels else "YOLO detected nothing on this frame")
show_detection_frame(frames, frame_idx, boxes, "1 - Lunar Lander YOLO Detection", labels=labels)
plot_detection_metric(lunar_yolo_counts, "1 - YOLO detections per frame", ylabel="detections")


## 2 - Car Racing Detection


In [ ]:
# YOLO decides what, if anything, it recognizes in CarRacing. No manual class labels.
if "detections_car" not in globals():
    detections_car = run_yolo(frames_car)
if "car_yolo_counts" not in globals():
    car_yolo_counts = yolo_detection_counts(detections_car)

boxes, labels = yolo_dets(detections_car[car_frame_idx])
print(labels if labels else "YOLO detected nothing on this frame")
show_detection_frame(frames_car, car_frame_idx, boxes, "2 - Car Racing YOLO Detection", labels=labels)
plot_detection_metric(car_yolo_counts, "2 - YOLO detections per frame", ylabel="detections")


## 3 - Recorded Traffic With People Detection


In [ ]:
if "detections_traffic" not in globals():
    detections_traffic = run_yolo(frames_medium)
if "traffic_yolo_counts" not in globals():
    traffic_yolo_counts = yolo_detection_counts(detections_traffic)

det_frame_idx = min(10, len(frames_medium) - 1)
boxes, labels = yolo_dets(detections_traffic[det_frame_idx])
print(labels if labels else "YOLO detected nothing on this frame")
show_detection_frame(frames_medium, det_frame_idx, boxes, "3 - Recorded Traffic With People YOLO Detection", labels=labels)
plot_detection_metric(traffic_yolo_counts, "3 - YOLO detections per frame", ylabel="detections")


## 4 - Random YouTube Driving Scene Detection


In [ ]:
if "detections_problem4" not in globals():
    detections_problem4 = run_yolo(frames_problem4)
if "problem4_yolo_counts" not in globals():
    problem4_yolo_counts = yolo_detection_counts(detections_problem4)

problem4_det_frame_idx = min(30, len(frames_problem4) - 1)
boxes, labels = yolo_dets(detections_problem4[problem4_det_frame_idx])
print(labels if labels else "YOLO detected nothing on this frame")
show_detection_frame(frames_problem4, problem4_det_frame_idx, boxes, "4 - Random YouTube Driving Scene YOLO Detection", labels=labels)
plot_detection_metric(problem4_yolo_counts, "4 - YOLO detections per sampled frame", ylabel="detections")


## 5 - Hard Vehicle-Crowd Interaction Detection


In [ ]:
if "detections_problem5" not in globals():
    detections_problem5 = run_yolo(frames_problem5)
if "problem5_yolo_counts" not in globals():
    problem5_yolo_counts = yolo_detection_counts(detections_problem5)

problem5_det_frame_idx = min(40, len(frames_problem5) - 1)
boxes, labels = yolo_dets(detections_problem5[problem5_det_frame_idx])
print(labels if labels else "YOLO detected nothing on this frame")
show_detection_frame(frames_problem5, problem5_det_frame_idx, boxes, "5 - Hard Vehicle-Crowd Interaction YOLO Detection", labels=labels)
plot_detection_metric(problem5_yolo_counts, "5 - YOLO detections per sampled frame", ylabel="detections")


# Object Segmentation


In [ ]:
# SAM-style class-agnostic object segmentation setup
# FastSAM returns mask proposals/blobs. We do not use semantic class labels here.
!pip install ultralytics -q

from ultralytics import FastSAM
import pandas as pd

seg_model = FastSAM("FastSAM-s.pt")

def run_segmentation(frame, imgsz=640, conf=0.35, iou=0.9):
    result = seg_model(frame, imgsz=imgsz, conf=conf, iou=iou, retina_masks=True, verbose=False)[0]
    if result.masks is None:
        return []
    masks = result.masks.data.cpu().numpy().astype(bool)
    return masks

def mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    area = int(mask.sum())
    centroid = (float(xs.mean()), float(ys.mean()))
    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": area,
        "centroid": centroid,
    }

def segmentation_records(frame, min_area=80, max_objects=25):
    masks = run_segmentation(frame)
    records = []
    for mask in masks:
        record = mask_to_record(mask, len(records))
        if record is not None and record["area"] >= min_area:
            records.append(record)
    records = sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
    for object_id, record in enumerate(records):
        record["object_id"] = object_id
    return records

def segmentation_summary(frames_batch, max_frames=40, stride=8, min_area=120):
    sampled = frames_batch[::stride][:max_frames]
    object_counts = []
    total_mask_area = []
    for frame in sampled:
        records = segmentation_records(frame, min_area=min_area)
        object_counts.append(len(records))
        total_mask_area.append(sum(obj["area"] for obj in records))
    return sampled, np.asarray(object_counts), np.asarray(total_mask_area)

def mask_boundary(mask):
    mask = mask.astype(bool)
    padded = np.pad(mask, 1, mode="constant", constant_values=False)
    center = padded[1:-1, 1:-1]
    eroded = (
        padded[:-2, 1:-1]
        & padded[2:, 1:-1]
        & padded[1:-1, :-2]
        & padded[1:-1, 2:]
        & center
    )
    return center & ~eroded

def show_segmentation_box_frame(frame, records, title):
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frame)
    for record in records:
        x1, y1, x2, y2 = record["box"]
        cx, cy = record["centroid"]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.8, edgecolor="lime")
        ax.add_patch(rect)
        ax.scatter([cx], [cy], s=18, c="yellow", edgecolors="black", linewidths=0.6)
        ax.text(x1, max(0, y1 - 4), f"box_{record['object_id']}", color="lime", fontsize=8, weight="bold")
    ax.set_title(title)
    ax.axis("off")
    plt.show()

def show_segmentation_blob_frame(frame, records, title, alpha=0.45):
    overlay = frame.copy()
    rng = np.random.default_rng(0)
    colors = []
    for record in records:
        color = rng.integers(40, 255, size=3)
        colors.append(color)
        mask = record["mask"]
        overlay[mask] = (overlay[mask] * (1 - alpha) + color * alpha).astype(np.uint8)
        overlay[mask_boundary(mask)] = color

    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(overlay)
    for record, color in zip(records, colors):
        cx, cy = record["centroid"]
        ax.scatter([cx], [cy], s=18, c=[color / 255], edgecolors="black", linewidths=0.6)
        ax.text(cx + 3, cy + 3, f"blob_{record['object_id']}", color="white", fontsize=8, weight="bold")
    ax.set_title(title)
    ax.axis("off")
    plt.show()

def plot_segmentation_summary(object_counts, total_mask_area, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(object_counts)
    axes[0].set_title("mask count")
    axes[0].set_xlabel("sampled frame")
    axes[0].set_ylabel("objects")
    axes[0].grid(alpha=0.25)
    axes[1].plot(total_mask_area)
    axes[1].set_title("total mask area")
    axes[1].set_xlabel("sampled frame")
    axes[1].set_ylabel("pixels")
    axes[1].grid(alpha=0.25)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def boundary_points(mask, max_points=24):
    boundary = mask_boundary(mask)
    ys, xs = np.nonzero(boundary)
    if len(xs) == 0:
        return []
    order = np.argsort(np.arctan2(ys - ys.mean(), xs - xs.mean()))
    xs, ys = xs[order], ys[order]
    if len(xs) > max_points:
        sample_idx = np.linspace(0, len(xs) - 1, max_points).astype(int)
        xs, ys = xs[sample_idx], ys[sample_idx]
    return [(int(x), int(y)) for x, y in zip(xs, ys)]

def segmentation_dataframe(records, representation="blob", max_boundary_points=24):
    rows = []
    for obj in records:
        row = {
            "object_id": obj["object_id"],
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
        }
        if representation == "box":
            x1, y1, x2, y2 = obj["box"]
            row.update({
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": x2 - x1,
                "height": y2 - y1,
            })
        elif representation == "blob":
            row.update({
                "mask_shape": obj["mask"].shape,
                "mask_pixels": int(obj["mask"].sum()),
                "boundary_sample": boundary_points(obj["mask"], max_points=max_boundary_points),
            })
        else:
            raise ValueError(representation)
        rows.append(row)
    return pd.DataFrame(rows)

def show_segmentation_data(records, representation, title):
    df = segmentation_dataframe(records, representation=representation)
    print(title)
    display(df)
    return df

def show_segmentation_experiment(frames_batch, frame_idx, title, prefix, view="blob", min_area=120):
    records = segmentation_records(frames_batch[frame_idx], min_area=min_area)
    print("objects:", len(records))
    print([
        {"object_id": obj["object_id"], "area": obj["area"], "centroid": obj["centroid"]}
        for obj in records[:10]
    ])
    if view == "box":
        show_segmentation_box_frame(frames_batch[frame_idx], records, title)
    elif view == "blob":
        show_segmentation_blob_frame(frames_batch[frame_idx], records, title)
    else:
        raise ValueError(view)
    sampled_frames, counts, areas = segmentation_summary(frames_batch, max_frames=40, stride=8, min_area=min_area)
    plot_segmentation_summary(counts, areas, title)
    data = show_segmentation_data(records, representation=view, title=f"{title} data representation")
    return records, sampled_frames, counts, areas, data


## Object Segmentation Box


### 1 - Lunar Lander Box


In [ ]:
seg_box_1_lunar_records, seg_box_1_lunar_frames, seg_box_1_lunar_counts, seg_box_1_lunar_areas, seg_box_1_lunar_data = show_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Segmentation Box",
    "seg_box_1_lunar",
    view="box",
    min_area=120,
)


### 2 - Car Racing Box


In [ ]:
seg_box_2_car_records, seg_box_2_car_frames, seg_box_2_car_counts, seg_box_2_car_areas, seg_box_2_car_data = show_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Segmentation Box",
    "seg_box_2_car",
    view="box",
    min_area=120,
)


### 3 - Recorded Traffic With People Box


In [ ]:
seg_box_3_traffic_records, seg_box_3_traffic_frames, seg_box_3_traffic_counts, seg_box_3_traffic_areas, seg_box_3_traffic_data = show_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Segmentation Box",
    "seg_box_3_traffic",
    view="box",
    min_area=120,
)


### 4 - Random YouTube Driving Scene Box


In [ ]:
seg_box_4_youtube_records, seg_box_4_youtube_frames, seg_box_4_youtube_counts, seg_box_4_youtube_areas, seg_box_4_youtube_data = show_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Segmentation Box",
    "seg_box_4_youtube",
    view="box",
    min_area=120,
)


### 5 - Hard Vehicle-Crowd Interaction Box


In [ ]:
seg_box_5_hard_records, seg_box_5_hard_frames, seg_box_5_hard_counts, seg_box_5_hard_areas, seg_box_5_hard_data = show_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Segmentation Box",
    "seg_box_5_hard",
    view="box",
    min_area=120,
)


## Object Segmentation Blob


### 1 - Lunar Lander Blob


In [ ]:
seg_blob_1_lunar_records, seg_blob_1_lunar_frames, seg_blob_1_lunar_counts, seg_blob_1_lunar_areas, seg_blob_1_lunar_data = show_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Segmentation Blob",
    "seg_blob_1_lunar",
    view="blob",
    min_area=120,
)


### 2 - Car Racing Blob


In [ ]:
seg_blob_2_car_records, seg_blob_2_car_frames, seg_blob_2_car_counts, seg_blob_2_car_areas, seg_blob_2_car_data = show_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Segmentation Blob",
    "seg_blob_2_car",
    view="blob",
    min_area=120,
)


### 3 - Recorded Traffic With People Blob


In [ ]:
seg_blob_3_traffic_records, seg_blob_3_traffic_frames, seg_blob_3_traffic_counts, seg_blob_3_traffic_areas, seg_blob_3_traffic_data = show_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Segmentation Blob",
    "seg_blob_3_traffic",
    view="blob",
    min_area=120,
)


### 4 - Random YouTube Driving Scene Blob


In [ ]:
seg_blob_4_youtube_records, seg_blob_4_youtube_frames, seg_blob_4_youtube_counts, seg_blob_4_youtube_areas, seg_blob_4_youtube_data = show_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Segmentation Blob",
    "seg_blob_4_youtube",
    view="blob",
    min_area=120,
)


### 5 - Hard Vehicle-Crowd Interaction Blob


In [ ]:
seg_blob_5_hard_records, seg_blob_5_hard_frames, seg_blob_5_hard_counts, seg_blob_5_hard_areas, seg_blob_5_hard_data = show_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Segmentation Blob",
    "seg_blob_5_hard",
    view="blob",
    min_area=120,
)
